# 📝 แบบฝึกหัด: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า embedding, score, และ agent response ที่ต้องตรงกับรหัสนักศึกษา

## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

## 🎓 กรอกข้อมูลนักศึกษา

In [ ]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = ''   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = ''   # เช่น '6512345678'
PHONE        = ''   # เช่น '081-234-5678'
LINE_ID      = ''   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [ ]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

---
## 🎯 ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน)

- รวมข้อความจากทุกไฟล์ใน `homework_data/`
- Chunk ด้วย `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- สร้าง Embedding ด้วย `intfloat/multilingual-e5-large`
- ค้นหาด้วย query: `MY_QUERY` (ที่สร้างจากรหัสนักศึกษา)
- เก็บลง Qdrant collection ชื่อ `f'hw_{STUDENT_ID}'`

**📝 รายงาน:**
1. ได้ทั้งหมดกี่ chunks?
2. Chunk ไหนมี similarity สูงสุด? (score ทศนิยม 4 ตำแหน่ง)
3. Top-3 ผลลัพธ์จาก Qdrant มี score เท่าไร?

In [ ]:
# ขั้นตอนที่ 1: เขียนโค้ดที่นี่

# 💡 Hint:
#   1. อ่านไฟล์จาก 'homework_data/' รวมเป็น text เดียว
#   2. from langchain_text_splitters import RecursiveCharacterTextSplitter
#   3. splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
#   4. chunks = splitter.split_text(all_text)
#   5. from sentence_transformers import SentenceTransformer
#   6. model = SentenceTransformer('intfloat/multilingual-e5-large')
#   7. passages = ['passage: ' + c for c in chunks]
#   8. embeddings = model.encode(passages)
#   9. query_emb = model.encode(f'query: {MY_QUERY}')
#  10. ใช้ cosine_similarity() หา chunk ที่คล้ายสุด
#  11. from qdrant_client import QdrantClient, models
#  12. สร้าง collection, upsert, query_points



# ✅ Self-check (uncomment หลังเขียนโค้ดเสร็จ)
# assert len(chunks) > 0, '❌ ยังไม่ได้ chunk'
# assert len(qdrant_results) == 3, '❌ ควรได้ top_k=3 จาก Qdrant'
# print(f'✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

---
## 🎯 ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน)

- ตั้งค่า Gemini API Key (Colab Secrets)
- สร้าง **Custom Tool** อย่างน้อย 1 ตัว (ห้ามซ้ำกับ BMI ในคาบ)
- สร้าง **Agent** ด้วย Google ADK ที่ใช้ Tool ได้
- ทดลองคุย → แสดงว่า Agent เรียก Tool ได้จริง

**📝 รายงาน:**
1. Tool ของคุณทำอะไร? (อธิบาย 1-2 ประโยค)
2. แสดง output ที่ Agent เรียก Tool สำเร็จ
3. ทำไม docstring ถึงสำคัญ? (อธิบาย 1-2 ประโยค)

In [ ]:
# ขั้นตอนที่ 2: เติมโค้ดในที่ว่าง

# ตั้งค่า API Key
import os
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 วาง API Key: ')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# ─── สร้าง Custom Tool (ห้ามใช้ BMI) ───
# 💡 ตัวอย่าง: แปลงอุณหภูมิ, คำนวณ GPA, คำนวณดอกเบี้ย ฯลฯ
def my_custom_tool(param1: float, param2: float) -> str:
    """อธิบายว่า tool นี้ทำอะไร (สำคัญมาก! Agent จะอ่าน docstring นี้)

    Args:
        param1: อธิบาย parameter ที่ 1
        param2: อธิบาย parameter ที่ 2
    """
    # TODO: เขียน logic ที่นี่
    result = 0  # แก้ตรงนี้
    return f'ผลลัพธ์: {result}'

tool = FunctionTool(my_custom_tool)

# ─── สร้าง Agent ───
my_agent = LlmAgent(
    name='my_assistant',
    model='gemini-2.5-flash',
    instruction='คุณเป็นผู้ช่วย... (แก้ให้ตรงกับ tool ของคุณ)',
    tools=[tool]
)

# ─── ทดสอบ ───
async def chat_with_agent(agent, message):
    runner = InMemoryRunner(agent=agent, app_name='homework')
    session = await runner.session_service.create_session(
        app_name='homework', user_id='student'
    )
    content = genai_types.Content(
        role='user', parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# ถามคำถามที่ต้องใช้ tool
answer = await chat_with_agent(my_agent, 'ใส่คำถามที่ต้องใช้ tool ของคุณ')
print(f'🤖 Agent: {answer}')

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent ไม่ตอบ'
print(f'✅ Step 2 passed!')

---
## 🎯 ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน)

- สร้าง **RAG Tool** ที่ค้นจาก Qdrant (ใช้ collection จากขั้นตอนที่ 1)
- สร้าง **RAG Agent** ที่ใช้ RAG Tool ตอบคำถาม
- ถามคำถาม 3 ข้อ (กำหนดให้) → บันทึกคำตอบ
- ให้คะแนนคำตอบด้วย **LLM-as-Judge** (ใช้ Gemini ให้คะแนน 1-5)

**คำถามที่ต้องถาม:**
```python
questions = [
    f'query: {MY_QUERY}',   # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]
```

**📝 รายงาน:**
1. คำตอบของ RAG Agent ต่อ 3 คำถาม
2. LLM-as-Judge ให้คะแนนเท่าไร? (1-5 ต่อข้อ)
3. อธิบาย: Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร? (2-3 ประโยค)

In [ ]:
# ขั้นตอนที่ 3: เติมโค้ดในที่ว่าง

# ─── A) สร้าง RAG Tool ───
def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ที่จัดเก็บใน Qdrant
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI, Machine Learning, NLP

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    # TODO: ใช้ qdrant.query_points() ค้นหาจาก collection ที่สร้างใน Step 1
    # results = qdrant.query_points(collection_name=..., query=..., limit=3)
    # return ผลลัพธ์เป็น string
    pass

# ─── B) สร้าง RAG Agent ───
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-2.5-flash',
    instruction='คุณเป็น AI ที่ตอบคำถามจากฐานความรู้ ใช้ tool search_knowledge ค้นหาข้อมูลก่อนตอบเสมอ ตอบเป็นภาษาไทย',
    tools=[FunctionTool(search_knowledge)]
)

# ─── C) ถาม 3 คำถาม ───
questions = [
    MY_QUERY,  # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]

rag_answers = []
for q in questions:
    ans = await chat_with_agent(rag_agent, q)
    rag_answers.append({'question': q, 'answer': ans})
    print(f'\n{"="*50}')
    print(f'❓ {q}')
    print(f'🤖 {ans}')

# ─── D) LLM-as-Judge ───
from google import genai
judge_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

# 📋 Template สำหรับ judge (ใช้ตามนี้ได้เลย)
JUDGE_PROMPT = '''คุณเป็นผู้ตรวจคุณภาพคำตอบ AI

คำถาม: {question}
คำตอบ: {answer}

ให้คะแนน 1-5 ตามเกณฑ์:
- 5 = ถูกต้อง ครบถ้วน อธิบายชัดเจน
- 4 = ถูกต้อง แต่ขาดรายละเอียดบางส่วน
- 3 = ถูกบางส่วน มีข้อผิดพลาดเล็กน้อย
- 2 = ตอบไม่ตรงประเด็น หรือผิดหลายจุด
- 1 = ผิดทั้งหมด หรือไม่ตอบ

ตอบ JSON: {{"score": 0, "reason": "..."}}
'''

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
for qa in rag_answers:
    prompt = JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer'])
    resp = judge_client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
    )
    result = json.loads(resp.text)
    print(f'  ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')

# ✅ Self-check
assert len(rag_answers) == 3, '❌ ต้องตอบ 3 คำถาม'
print(f'\n✅ Step 3 passed: ตอบครบ 3 ข้อ + LLM-as-Judge เสร็จ!')

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงานได้, ผล Qdrant ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent ตอบครบ 3 ข้อ, LLM-as-Judge ให้คะแนน, อธิบาย |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: _________________ (อาจารย์กำหนด)')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน')
print('  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน')
print('  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')